# Real-Time ROI Sketching from Webcam Feed

In this notebook, we capture a live webcam feed, allow the user to **select a rectangular region of interest (ROI)** with the mouse, and then apply a **sketch effect** to that region in real-time.  

This project demonstrates **OpenCV’s ROI selection, image processing, and real-time video manipulation**.

---

### Key Steps:
1. Capture live video feed from webcam.
2. Allow user to draw a rectangle over the frame to define ROI.
3. Convert the selected ROI to a **sketch-style image**.
4. Replace the ROI in the original frame with the sketched version.
5. Stream the modified live feed continuously.

---

# Code Implementation


In [1]:
import cv2
import numpy as np

# Function to apply sketch effect to an image
def sketch_transform(image):
    # Convert to grayscale
    image_grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Apply Gaussian blur to smoothen edges
    image_grayscale_blurred = cv2.GaussianBlur(image_grayscale, (7,7), 0)
    # Apply Canny edge detection
    image_canny = cv2.Canny(image_grayscale_blurred, 10, 80)
    # Invert edges to create sketch effect
    _, mask = cv2.threshold(image_canny, 30, 255, cv2.THRESH_BINARY_INV)
    return mask

# Initialize webcam
cam_capture = cv2.VideoCapture(0)
cv2.destroyAllWindows()

# Let user select ROI on first frame
while True:
    _, im0 = cam_capture.read()
    showCrosshair = False
    fromCenter = False
    r = cv2.selectROI("Select ROI", im0, fromCenter, showCrosshair)
    break  # exit after selecting ROI

# Process live feed
while True:   
    _, image_frame = cam_capture.read()
    
    # Extract the selected ROI from current frame
    rect_img = image_frame[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]
    
    # Apply sketch effect to the ROI
    sketcher_rect = sketch_transform(rect_img)
    
    # Convert single channel back to 3-channel for overlay
    sketcher_rect_rgb = cv2.cvtColor(sketcher_rect, cv2.COLOR_GRAY2RGB)
    
    # Replace the ROI in the original frame with the sketched version
    image_frame[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])] = sketcher_rect_rgb
    
    # Display the live feed with sketched ROI
    cv2.imshow("Sketcher ROI", image_frame)
    
    # Exit on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
# Release webcam and destroy all windows
cam_capture.release()
cv2.destroyAllWindows()


# Basic Image Input/Output with OpenCV

This example demonstrates how to **read, display, and save images** using OpenCV.  

We also explore **image dimensions** and how the image array is structured.

---

### Key Steps:
1. Read an image from a file using `cv2.imread()`.
2. Display the image with `cv2.imshow()`.
3. Inspect the **shape** of the image array to understand its dimensions.
4. Save the image in different formats (`.jpg`, `.png`) using `cv2.imwrite()`.

---

# Code Implementation


In [2]:
import numpy as np
import cv2

# Read the input image
input_img = cv2.imread("IMG_0213.jpg")

# Display the input image
cv2.imshow('Input Image', input_img)
cv2.waitKey(0)  # Wait until a key is pressed
cv2.destroyAllWindows()

# Inspect the dimensions of the image
height, width, channels = input_img.shape
print(f"Height of Image: {height} pixels")
print(f"Width of Image: {width} pixels")
print(f"Number of Channels: {channels} (RGB)")

# Save the image in different formats
cv2.imwrite('output.jpg', input_img)
cv2.imwrite('output.png', input_img)


Height of Image: 685 pixels
Width of Image: 701 pixels
Number of Channels: 3 (RGB)


True

# Car Detection using Haar Cascade in OpenCV

This notebook demonstrates **vehicle detection** in videos using OpenCV's Haar Cascade Classifier.

---

### Steps Covered:
1. Load the Haar Cascade classifier for car detection.
2. Load a video and process it frame by frame.
3. Convert each frame to grayscale for better detection.
4. Detect cars using `detectMultiScale`.
5. Draw rectangles around detected cars.
6. Display the real-time detection and handle exit keys.

---

# Code Implementation


In [1]:
import cv2
import time

# Path to the Haar Cascade for car detection
car_classifier_path = 'haarcascade_car.xml'

# Load the car classifier
car_classifier = cv2.CascadeClassifier(car_classifier_path)

# Check if the classifier loaded successfully
if car_classifier.empty():
    print(f"Error: Could not load the car classifier at {car_classifier_path}")
    exit()

# Path to the video file
video_path = 'cars.mp4'

# Load the video
cap = cv2.VideoCapture(video_path)

# Check if the video opened successfully
if not cap.isOpened():
    print(f"Error: Could not open the video at {video_path}")
    exit()

print("Video opened successfully. Starting car detection...")

# Process the video frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Video ended or failed to capture frame.")
        break

    # Convert frame to grayscale for Haar detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect cars
    cars = car_classifier.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))

    # Draw rectangles around detected cars
    for (x, y, w, h) in cars:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 255), 2)

    # Display the frame with detected cars
    cv2.imshow('Car Detection', frame)

    # Exit if Enter key is pressed
    if cv2.waitKey(1) == 13:  # 13 is Enter key
        print("Exiting...")
        break

    # Optional: small delay to smooth processing
    time.sleep(0.05)

# Release resources
cap.release()
cv2.destroyAllWindows()


Video opened successfully. Starting car detection...
Exiting...


# Real-Time Face and Eye Detection using Haar Cascades

This notebook demonstrates **real-time face and eye detection** using OpenCV's Haar Cascade Classifiers.

---

### Features:
- Detect faces in webcam feed using Haar Cascade.
- Detect eyes within the detected face regions.
- Draw rectangles around faces and eyes for visualization.
- Object-oriented implementation for modularity and reusability.

---

# Code Implementation


In [2]:
import cv2

class FaceAndEyeDetection:
    def __init__(self, face_cascade_path, eye_cascade_path):
        # Load the cascade classifiers for face and eyes
        self.face_cascade = cv2.CascadeClassifier(face_cascade_path)
        self.eye_cascade = cv2.CascadeClassifier(eye_cascade_path)
        
        # Check if classifiers were loaded properly
        if self.face_cascade.empty():
            raise IOError("Error: Could not load face cascade.")
        if self.eye_cascade.empty():
            raise IOError("Error: Could not load eye cascade.")

    def detect_faces(self, gray, frame):
        """Detect faces in the given frame."""
        faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
        for (x, y, w, h) in faces:
            # Draw rectangle around the face
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            
            # Region of interest for eyes
            roi_gray = gray[y:y + h, x:x + w]
            roi_color = frame[y:y + h, x:x + w]
            
            # Detect eyes inside the face ROI
            self.detect_eyes(roi_gray, roi_color)
        return frame

    def detect_eyes(self, roi_gray, roi_color):
        """Detect eyes in the region of interest (ROI)."""
        eyes = self.eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=5)
        for (ex, ey, ew, eh) in eyes:
            # Draw rectangle around each eye
            cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)

    def start_detection(self):
        """Start webcam capture and detect faces and eyes in real-time."""
        video_capture = cv2.VideoCapture(0)

        if not video_capture.isOpened():
            print("Error: Could not access the webcam.")
            return

        while True:
            ret, frame = video_capture.read()
            if not ret:
                print("Error: Failed to capture image.")
                break

            # Convert frame to grayscale for Haar detection
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Detect faces and eyes
            canvas = self.detect_faces(gray, frame)

            # Display the results
            cv2.imshow('Face and Eye Detection', canvas)

            # Press 'q' to quit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Exiting...")
                break

        video_capture.release()
        cv2.destroyAllWindows()


# ✅ Correct Haar cascade paths
# OpenCV includes them inside cv2.data.haarcascades
face_cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
eye_cascade_path = cv2.data.haarcascades + 'haarcascade_eye.xml'

# Initialize the detection system
detection_system = FaceAndEyeDetection(face_cascade_path, eye_cascade_path)

# Start face and eye detection
detection_system.start_detection()


Exiting...


# Real-Time Face and Eye Detection using Haar Cascades (Simple Version)

This notebook demonstrates **real-time face and eye detection** using OpenCV's Haar Cascade classifiers.

---

### Features:
- Detect faces in webcam feed.
- Detect eyes within detected face regions.
- Draw rectangles around faces and eyes for visualization.
- Simple functional approach without classes.

---

# Code Implementation

In [1]:
import cv2

# Load Haar cascade files for face and eye detection
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier("haarcascade_eye.xml")

# Check if cascade files are loaded properly
if face_cascade.empty():
    print("Error: Could not load face cascade classifier.")
    exit()
if eye_cascade.empty():
    print("Error: Could not load eye cascade classifier.")
    exit()

# Function to detect faces and eyes
def detect_faces_and_eyes(gray, frame):
    # Detect faces in grayscale frame
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=6, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE)
    
    for (x, y, w, h) in faces:
        # Draw rectangle around face
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        
        # Region of interest for eyes
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = frame[y:y + h, x:x + w]
        
        # Detect eyes in the face region
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            # Draw rectangle around eyes
            cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)
    
    return frame

# Initialize webcam capture
video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    print("Error: Could not access the webcam.")
    exit()

print("Webcam opened successfully. Starting face and eye detection...")

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break

    # Convert frame to grayscale for detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces and eyes
    result_frame = detect_faces_and_eyes(gray, frame)

    # Display the frame
    cv2.imshow('Face and Eye Detection', result_frame)

    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting...")
        break

# Release resources
video_capture.release()
cv2.destroyAllWindows()


Webcam opened successfully. Starting face and eye detection...
Exiting...


# Face and Eye Detection on a Static Image using Haar Cascades

This example demonstrates detecting faces and eyes in a **static image** using OpenCV's Haar Cascade classifiers.

---

### Features:
- Load a static image.
- Detect faces in the image.
- Detect eyes within the detected face regions.
- Draw rectangles around faces and eyes for visualization.


In [2]:
import numpy as np
import cv2

# Load Haar cascade classifiers for face and eyes
face_classifier = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
eye_classifier = cv2.CascadeClassifier("haarcascade_eye.xml")

# Load the image
img_path = "D:\MLOPS\IMG_0213.jpg"
img = cv2.imread(img_path)

# Check if the image is loaded correctly
if img is None:
    print("Error: Image not found or cannot be loaded!")
    exit()

# Convert image to grayscale for face detection
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Detect faces in the image
faces = face_classifier.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

# Check if any faces are detected
if len(faces) == 0:
    print("No Face Found")

# Draw rectangles around detected faces and detect eyes within each face
for (x, y, w, h) in faces:
    # Draw rectangle around the face
    cv2.rectangle(img, (x, y), (x + w, y + h), (127, 0, 255), 2)
    
    # Region of interest for face
    roi_gray = gray[y:y + h, x:x + w]
    roi_color = img[y:y + h, x:x + w]

    # Detect eyes in the face region
    eyes = eye_classifier.detectMultiScale(roi_gray)
    for (ex, ey, ew, eh) in eyes:
        # Draw rectangle around each detected eye
        cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (255, 255, 0), 2)

# Display the result
cv2.imshow('Face and Eye Detection', img)

# Wait for a key press to close
cv2.waitKey(0)
cv2.destroyAllWindows()


<>:9: SyntaxWarning: invalid escape sequence '\M'
<>:9: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Palakolanu Mounika\AppData\Local\Temp\ipykernel_14676\1815745610.py:9: SyntaxWarning: invalid escape sequence '\M'
  img_path = "D:\MLOPS\IMG_0213.jpg"


# Real-Time Face and Eye Detection using Webcam

This example demonstrates detecting faces and eyes in **real-time** using OpenCV's Haar Cascade classifiers via webcam feed.

---

### Features:
- Capture live video from webcam.
- Detect faces in each frame.
- Detect eyes within the detected faces.
- Draw rectangles around detected faces (blue) and eyes (green).
- Press 'q' to quit the program.


In [3]:
import cv2

# Load Haar cascade classifiers for face and eyes
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier("haarcascade_eye.xml")

# Verify if the classifiers were loaded correctly
if face_cascade.empty():
    print("Error: Could not load face cascade classifier.")
    exit()
if eye_cascade.empty():
    print("Error: Could not load eye cascade classifier.")
    exit()

# Function to detect faces and eyes in a frame
def detect_faces_and_eyes(gray, frame):
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    
    for (x, y, w, h) in faces:
        # Draw rectangle around the face
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        
        # Region of interest for eyes detection
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = frame[y:y + h, x:x + w]
        
        # Detect eyes in the face region
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            # Draw rectangle around each eye
            cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)

    return frame

# Initialize webcam capture
video_capture = cv2.VideoCapture(0)

# Check if webcam opened correctly
if not video_capture.isOpened():
    print("Error: Could not access the webcam.")
    exit()

print("Webcam opened successfully. Starting face and eye detection...")

while True:
    # Capture frame-by-frame
    ret, frame = video_capture.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break

    # Convert frame to grayscale for detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces and eyes
    result_frame = detect_faces_and_eyes(gray, frame)

    # Display the result
    cv2.imshow('Face and Eye Detection', result_frame)

    # Exit loop on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting...")
        break

# Release resources
video_capture.release()
cv2.destroyAllWindows()


Webcam opened successfully. Starting face and eye detection...
Exiting...


# Pedestrian Detection using Haar Cascade

This notebook demonstrates detecting pedestrians (full bodies) in a video using OpenCV's **Haar Cascade classifier**.

---

### Features:
- Load a video file containing pedestrians.
- Detect full bodies in each frame.
- Draw bounding boxes around detected pedestrians.
- Press **Enter** to exit the program.


In [4]:
import cv2
import numpy as np
import os

# Path to Haar Cascade for full body detection
body_classifier_path = 'haarcascade_fullbody.xml'

# Verify if the classifier file exists
if not os.path.exists(body_classifier_path):
    print(f"Error: The classifier file does not exist at {body_classifier_path}")
    exit()

# Load the body classifier
body_classifier = cv2.CascadeClassifier(body_classifier_path)

# Verify if classifier is loaded correctly
if body_classifier.empty():
    print("Error: Could not load the body classifier. Ensure the XML file is valid.")
    exit()

# Path to the video file
video_path = 'Walking.mp4'

# Check if the video file exists
if not os.path.exists(video_path):
    print(f"Error: The video file does not exist at {video_path}")
    exit()

# Open the video
cap = cv2.VideoCapture(video_path)

# Check if video opened successfully
if not cap.isOpened():
    print(f"Error: Could not open video file at {video_path}")
    exit()

print("Video opened successfully. Starting pedestrian detection...")

while cap.isOpened():
    # Read frame from video
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to read frame from video. Exiting...")
        break

    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect bodies in the grayscale frame
    bodies = body_classifier.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=3,
        minSize=(50, 50),
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    # Draw rectangles around detected bodies
    for (x, y, w, h) in bodies:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 255), 2)

    # Display the frame with detections
    cv2.imshow('Pedestrians', frame)

    # Exit on pressing Enter (key code 13)
    if cv2.waitKey(1) == 13:
        print("Exiting...")
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


Video opened successfully. Starting pedestrian detection...
Error: Failed to read frame from video. Exiting...
